# **Ejercicio 6 — Modelos de clasificación**

## **Enfoque**

Siguiendo un enfoque similar al visto en clase, no se trata solo de probar varios algoritmos y
quedarse con el que dé el número más alto, sino de **validar qué combinación de representación del
texto y de modelo funciona mejor para este dataset en particular**. Es por esto que cada modelo se
corre sobre la misma rejilla de representaciones y se compara todo contra la misma partición de
prueba.

Los tres modelos que se prueban son:

- **Regresión logística**, como baseline lineal. Es el punto de comparación obligado, ya que si un
  modelo complejo no le gana a una regresión sobre bolsa de palabras, entonces la complejidad no se
  justifica.
- **Random Forest**, para poder tener un modelo complejo basado en árboles. Se usa con validación
  cruzada sobre el conjunto de entrenamiento, buscando garantizar que lo que aprende son patrones y
  no particularidades de una sola partición.
- **Red neuronal con LSTM**, aprovechando lo trabajado en el curso de deep learning, para poder
  identificar si la secuencia específica de palabras aporta algo que los modelos de bolsa de
  palabras no capturan.

Cada modelo se prueba variando cuatro cosas:

- **Reducción de la palabra:** lematización contra stemming.
- **N-gramas:** unigramas contra unigramas más bigramas.
- **Vectorización:** bolsa de palabras contra TF-IDF.

Cabe mencionar que la LSTM no entra en la rejilla completa, ya que trabaja sobre secuencias de
índices con una capa de embeddings y no sobre una matriz de bolsa de palabras, de tal forma que en
su caso solo varía la reducción de la palabra.

In [12]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import cross_val_score

SEMILLA = 123
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)

train = pd.read_csv("../data/processed/train_modelado.csv")
test = pd.read_csv("../data/processed/test_modelado.csv")
for d in (train, test):
    d["tokens"] = d["tokens"].fillna("")
    d["tokens_stem"] = d["tokens_stem"].fillna("")

y_train, y_test = train["target"].values, test["target"].values

COLUMNA = {"lematizacion": "tokens", "stemming": "tokens_stem"}
NGRAMA = {"unigrama": (1, 1), "unigrama+bigrama": (1, 2)}
VECTORIZADOR = {"bag of words": CountVectorizer, "tf-idf": TfidfVectorizer}

resultados = []
print(f"entrenamiento: {len(train):,} | prueba: {len(test):,}")

entrenamiento: 5,239 | prueba: 2,246


In [13]:
def metricas(y_real, y_pred):
    return {
        "accuracy": accuracy_score(y_real, y_pred),
        "precision": precision_score(y_real, y_pred),
        "recall": recall_score(y_real, y_pred),
        "f1": f1_score(y_real, y_pred),
    }


def registrar(modelo, texto, ngrama, vector, y_pred, cv=None):
    m = metricas(y_test, y_pred)
    resultados.append({"modelo": modelo, "texto": texto, "ngrama": ngrama,
                       "vectorizador": vector, "cv_f1": cv, **m})
    cv_txt = f" | cv_f1 {cv:.4f}" if cv is not None else ""
    print(f"  {texto:13s} {ngrama:17s} {vector:12s} -> "
          f"accuracy {m['accuracy']:.4f} | precision {m['precision']:.4f} | "
          f"recall {m['recall']:.4f} | f1 {m['f1']:.4f}{cv_txt}")


def vectorizar(texto, ngrama, vector):
    Vec = VECTORIZADOR[vector]
    v = Vec(ngram_range=NGRAMA[ngrama], min_df=2)
    X_tr = v.fit_transform(train[COLUMNA[texto]])
    X_te = v.transform(test[COLUMNA[texto]])
    return X_tr, X_te

In [14]:
print("REGRESION LOGISTICA (baseline lineal)")
for texto in COLUMNA:
    for ngrama in NGRAMA:
        for vector in VECTORIZADOR:
            X_tr, X_te = vectorizar(texto, ngrama, vector)
            modelo = LogisticRegression(max_iter=1000, random_state=SEMILLA)
            modelo.fit(X_tr, y_train)
            registrar("Regresion logistica", texto, ngrama, vector, modelo.predict(X_te))

REGRESION LOGISTICA (baseline lineal)
  lematizacion  unigrama          bag of words -> accuracy 0.7983 | precision 0.8014 | recall 0.7001 | f1 0.7474
  lematizacion  unigrama          tf-idf       -> accuracy 0.8028 | precision 0.8278 | recall 0.6782 | f1 0.7455
  lematizacion  unigrama+bigrama  bag of words -> accuracy 0.7965 | precision 0.8041 | recall 0.6907 | f1 0.7431
  lematizacion  unigrama+bigrama  tf-idf       -> accuracy 0.8050 | precision 0.8314 | recall 0.6803 | f1 0.7483
  stemming      unigrama          bag of words -> accuracy 0.7956 | precision 0.7971 | recall 0.6980 | f1 0.7443
  stemming      unigrama          tf-idf       -> accuracy 0.8077 | precision 0.8269 | recall 0.6938 | f1 0.7545
  stemming      unigrama+bigrama  bag of words -> accuracy 0.7996 | precision 0.8080 | recall 0.6949 | f1 0.7472
  stemming      unigrama+bigrama  tf-idf       -> accuracy 0.8063 | precision 0.8263 | recall 0.6907 | f1 0.7524


In [15]:
print("RANDOM FOREST (con validacion cruzada de 5 pliegues sobre entrenamiento)")
for texto in COLUMNA:
    for ngrama in NGRAMA:
        for vector in VECTORIZADOR:
            X_tr, X_te = vectorizar(texto, ngrama, vector)
            modelo = RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                            n_jobs=-1, random_state=SEMILLA)
            cv = cross_val_score(modelo, X_tr, y_train, cv=5, scoring="f1", n_jobs=-1).mean()
            modelo.fit(X_tr, y_train)
            registrar("Random Forest", texto, ngrama, vector, modelo.predict(X_te), cv)

RANDOM FOREST (con validacion cruzada de 5 pliegues sobre entrenamiento)
  lematizacion  unigrama          bag of words -> accuracy 0.8059 | precision 0.7984 | recall 0.7283 | f1 0.7617 | cv_f1 0.7373
  lematizacion  unigrama          tf-idf       -> accuracy 0.7930 | precision 0.7697 | recall 0.7335 | f1 0.7512 | cv_f1 0.7232
  lematizacion  unigrama+bigrama  bag of words -> accuracy 0.8041 | precision 0.8081 | recall 0.7085 | f1 0.7550 | cv_f1 0.7343
  lematizacion  unigrama+bigrama  tf-idf       -> accuracy 0.8023 | precision 0.7918 | recall 0.7273 | f1 0.7582 | cv_f1 0.7274
  stemming      unigrama          bag of words -> accuracy 0.7925 | precision 0.7832 | recall 0.7095 | f1 0.7445 | cv_f1 0.7374
  stemming      unigrama          tf-idf       -> accuracy 0.7943 | precision 0.7765 | recall 0.7262 | f1 0.7505 | cv_f1 0.7327
  stemming      unigrama+bigrama  bag of words -> accuracy 0.8001 | precision 0.8031 | recall 0.7032 | f1 0.7499 | cv_f1 0.7358
  stemming      unigrama+bigram

In [16]:
MAX_LONGITUD, DIM_EMBEDDING, UNIDADES, EPOCAS, LOTE = 24, 64, 64, 8, 64


class RedLSTM(nn.Module):
    def __init__(self, n_vocabulario):
        super().__init__()
        self.embedding = nn.Embedding(n_vocabulario, DIM_EMBEDDING, padding_idx=0)
        self.lstm = nn.LSTM(DIM_EMBEDDING, UNIDADES, batch_first=True)
        self.dropout = nn.Dropout(0.3)
        self.salida = nn.Linear(UNIDADES, 1)

    def forward(self, x):
        _, (h, _) = self.lstm(self.embedding(x))
        return self.salida(self.dropout(h[-1])).squeeze(1)


def secuenciar(serie, indice):
    X = np.zeros((len(serie), MAX_LONGITUD), dtype=np.int64)
    for i, texto in enumerate(serie):
        ids = [indice[t] for t in texto.split()[:MAX_LONGITUD] if t in indice]
        X[i, :len(ids)] = ids
    return torch.tensor(X)


print("RED NEURONAL LSTM")
for texto in COLUMNA:
    col = COLUMNA[texto]
    vocabulario = {t for s in train[col] for t in s.split()}
    indice = {t: i + 1 for i, t in enumerate(sorted(vocabulario))}

    X_tr, X_te = secuenciar(train[col], indice), secuenciar(test[col], indice)
    y_tr = torch.tensor(y_train, dtype=torch.float32)

    torch.manual_seed(SEMILLA)
    red = RedLSTM(len(indice) + 1)
    optimizador = torch.optim.Adam(red.parameters(), lr=1e-3)
    perdida = nn.BCEWithLogitsLoss()

    red.train()
    for _ in range(EPOCAS):
        orden = torch.randperm(len(X_tr))
        for i in range(0, len(orden), LOTE):
            lote = orden[i:i + LOTE]
            optimizador.zero_grad()
            error = perdida(red(X_tr[lote]), y_tr[lote])
            error.backward()
            optimizador.step()

    red.eval()
    with torch.no_grad():
        y_pred = (torch.sigmoid(red(X_te)) >= 0.5).int().numpy()
    registrar("LSTM", texto, "secuencia", "embeddings", y_pred)

RED NEURONAL LSTM
  lematizacion  secuencia         embeddings   -> accuracy 0.7689 | precision 0.7731 | recall 0.6479 | f1 0.7049
  stemming      secuencia         embeddings   -> accuracy 0.7743 | precision 0.7812 | recall 0.6531 | f1 0.7114


In [17]:
tabla = pd.DataFrame(resultados).sort_values("f1", ascending=False).reset_index(drop=True)
tabla.round(4)

,modelo,texto,ngrama,vectorizador,cv_f1,accuracy,precision,recall,f1
0,Random Forest,lematizacion,unigrama,bag of words,0.7373,0.8059,0.7984,0.7283,0.7617
1,Random Forest,lematizacion,unigrama+bigrama,tf-idf,0.7274,0.8023,0.7918,0.7273,0.7582
2,Random Forest,lematizacion,unigrama+bigrama,bag of words,0.7343,0.8041,0.8081,0.7085,0.7550
3,Regresion logistica,stemming,unigrama,tf-idf,NaN,0.8077,0.8269,0.6938,0.7545
4,Regresion logistica,stemming,unigrama+bigrama,tf-idf,NaN,0.8063,0.8262,0.6907,0.7524
5,Random Forest,lematizacion,unigrama,tf-idf,0.7232,0.7930,0.7697,0.7335,0.7512
6,Random Forest,stemming,unigrama,tf-idf,0.7327,0.7943,0.7765,0.7262,0.7505
7,Random Forest,stemming,unigrama+bigrama,bag of words,0.7358,0.8001,0.8031,0.7032,0.7499
8,Regresion logistica,lematizacion,unigrama+bigrama,tf-idf,NaN,0.8050,0.8314,0.6803,0.7483
9,Random Forest,stemming,unigrama+bigrama,tf-idf,0.7273,0.7947,0.7838,0.7158,0.7482


In [18]:
print("MEJOR CONFIGURACION POR MODELO\n")
mejores = tabla.loc[tabla.groupby("modelo")["f1"].idxmax()].sort_values("f1", ascending=False)
print(mejores[["modelo", "texto", "ngrama", "vectorizador",
               "accuracy", "precision", "recall", "f1"]].round(4).to_string(index=False))

print("\n\nPROMEDIO DE F1 POR CADA OPCION DE REPRESENTACION\n")
for columna in ("texto", "ngrama", "vectorizador"):
    print(tabla.groupby(columna)["f1"].mean().round(4).to_string(), "\n")

MEJOR CONFIGURACION POR MODELO

             modelo        texto    ngrama vectorizador  accuracy  precision  recall     f1
      Random Forest lematizacion  unigrama bag of words    0.8059     0.7984  0.7283 0.7617
Regresion logistica     stemming  unigrama       tf-idf    0.8077     0.8269  0.6938 0.7545
               LSTM     stemming secuencia   embeddings    0.7743     0.7812  0.6531 0.7114


PROMEDIO DE F1 POR CADA OPCION DE REPRESENTACION

texto
lematizacion    0.7462
stemming        0.7448 

ngrama
secuencia           0.7082
unigrama            0.7500
unigrama+bigrama    0.7503 

vectorizador
bag of words    0.7491
embeddings      0.7082
tf-idf          0.7511 



## **Conclusiones**

El mejor resultado por f1 lo dio el **Random Forest con lematización, unigramas y bolsa de
palabras**, con un f1 de 0.7617 y una accuracy de 0.8059. Muy cerca quedó la regresión logística con
stemming, unigramas y TF-IDF, con un f1 de 0.7545 y de hecho la accuracy más alta de toda la tabla,
0.8077.

**Lo primero que hay que decir es que la diferencia entre modelos es frágil.** Las 18
configuraciones caen entre 0.7049 y 0.7617 de f1, y los dos primeros puestos están separados por
0.0072. Cabe mencionar que en una corrida previa con otra partición el orden estaba invertido y era
la regresión logística la que encabezaba, o sea que **la distancia entre el primero y el segundo es
menor que la variación que produce cambiar la partición**. Bajo esta idea, considero que lo honesto
no es declarar un ganador tajante entre esos dos, sino reconocer que ambos rinden igual dentro del
margen de error, y que la elección entre ellos debería apoyarse en otros criterios además del f1.

Con 0.7114 usando stemming y 0.7049 usando lematización, se queda entre 0.04 y 0.05 por debajo del resto, que es una brecha mucho
mayor que la que separa a los otros dos modelos entre sí. Diría que la causa es la cantidad de datos
frente a lo que el modelo tiene que aprender, ya que una LSTM debe estimar los embeddings y los pesos
recurrentes desde cero con apenas 5,239 ejemplos de unos 8 tokens cada uno. Esto además confirma lo
que se midió en el ejercicio 4, donde se vio que los n-gramas largos casi no se repiten: si el orden
de las palabras aportara señal aprovechable, la LSTM debería haberla capturado, y no lo hizo.

**Regresión logística contra Random Forest: el reparto entre precisión y recall.** Acá está la
diferencia real entre los dos, y no en el f1:

- La regresión logística es más conservadora, con precisiones entre 0.797 y 0.831 y recalls entre
  0.678 y 0.700.
- El Random Forest es más agresivo, con precisiones entre 0.770 y 0.808 pero recalls entre 0.703 y
  0.734.

Es decir que el Random Forest detecta más desastres reales a costa de equivocarse más seguido, y en
este problema me parece la compensación correcta, ya que dejar pasar un desastre real es más costoso
que marcar de más un tweet que no lo era.

**Sobre las representaciones**:

- Lematización y stemming quedaron prácticamente empatados, 0.7462 contra 0.7448. La lematización
  ganó en el mejor Random Forest y el stemming en la mejor regresión logística, de tal forma que no
  hay una opción claramente superior.
- TF-IDF quedó levemente arriba de la bolsa de palabras, 0.7511 contra 0.7491, aunque la mejor
  configuración de todas usó bolsa de palabras.
- Agregar bigramas quedó en empate técnico, 0.7503 contra 0.7500 de los unigramas solos. No obstante,
  como los bigramas multiplican el tamaño de la matriz sin dar nada a cambio, **se prefieren los
  unigramas**, lo cual coincide con lo que se había concluido en el ejercicio 4 al ver que solo el
  11% de los bigramas aparecía en dos o más tweets.
- Las secuencias con embeddings quedaron muy por detrás, con 0.7082 de promedio.

**El patrón que atraviesa toda la tabla.** En las 18 configuraciones la precisión supera al recall,
o sea que cuando el clasificador dice que un tweet habla de un desastre suele acertar, sin embargo
se le escapa entre un 27% y un 35% de los desastres reales. Tiene sentido dado lo visto en el
análisis exploratorio, ya que el vocabulario de desastre es muy variado mientras que el de la clase
contraria es más repetitivo. Para poder mejorar el recall convendría ajustar el umbral de decisión o
incorporar las variables derivadas del ejercicio 3, sobre todo `tiene_url`, que por sí sola separaba
un 30% contra un 55%.

**Modelo seleccionado:** Random Forest con lematización, unigramas y bolsa de palabras, por tener el
mejor f1 y el mejor balance entre precisión y recall. Es el que se usará en la función de
clasificación del ejercicio 7 y el que se reentrenará con la variable de negatividad en el ejercicio
10. 